[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burzynski-lab/OU_PHYS2222_ComputationalPhysics/blob/main/LectureNotes/17_Neural_Networks.ipynb)

# Machine Learning II: Neural Networks with PyTorch

**PHYS 2222: Computational Physics** · University of Oklahoma · Spring 2027 · Week 14

> **Before you start:** in Colab, choose **File → Save a copy in Drive** so your changes are kept.

Last week you saw that machine learning means "a flexible model with many parameters, fit by minimizing a loss". This week's model is the **neural network**. Despite the name, a neural network is just a particular kind of function, built by stacking many copies of a very simple building block. What makes neural networks special is that the same recipe works for fitting a smooth curve, recognizing an image, or (scaled up enormously) generating text.

You will build one from scratch in NumPy so that nothing is hidden, and then use **PyTorch**, the library most physicists (and most of the ML world) use for neural networks.

## Learning goals

After this notebook you should be able to:

- Describe a neuron (weighted sum + activation function), a layer, and a multilayer perceptron (MLP).
- Write down the mean-squared-error and binary cross-entropy losses and say when each is used.
- Explain gradient descent with a picture, and what happens when the learning rate is too small or too large.
- Train a tiny network in pure NumPy, computing the gradients by hand with the chain rule.
- Use PyTorch tensors and **autograd** to compute derivatives automatically.
- Build networks with `nn.Sequential`, train them with an optimizer (Adam) and mini-batches (`DataLoader`), and monitor training and validation loss.
- Recognize overfitting in loss curves and apply early stopping.
- Explain why a neural network fails to extrapolate outside its training range.
- Compare a neural network with gradient-boosted trees on the MAGIC data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import constants

rng = np.random.default_rng(14)

## 1. The neuron

A single artificial **neuron** takes some inputs $x_1, \dots, x_d$, multiplies each by a **weight** $w_j$, adds a **bias** $b$, and passes the result through a nonlinear **activation function** $\sigma$:

$$a = \sigma\big(w_1x_1 + w_2x_2 + \dots + w_dx_d + b\big) = \sigma(\mathbf{w}\cdot\mathbf{x} + b).$$

(Reminder from last week: $\mathbf{w}\cdot\mathbf{x}$ is the dot product, the sum of products of matching entries.) The weights and bias are the parameters that get fit. If you drop the activation, a neuron is exactly linear regression; with a sigmoid activation, it is exactly logistic regression. So you have already trained single neurons.

Common activation functions:

- **Sigmoid** $\sigma(z) = 1/(1+e^{-z})$: squashes to (0, 1). Used at the output to produce a probability.
- **tanh** $\tanh z$: squashes to (−1, 1). Smooth, zero-centered.
- **ReLU** (rectified linear unit) $\max(0, z)$: zero for negative input, identity for positive. Simple, fast, and the most common choice in hidden layers of large networks.

In [ ]:
z = np.linspace(-5, 5, 400)

fig, ax = plt.subplots()
ax.plot(z, 1 / (1 + np.exp(-z)), label=r"sigmoid $1/(1+e^{-z})$")
ax.plot(z, np.tanh(z), label=r"$\tanh z$")
ax.plot(z, np.maximum(0, z), label=r"ReLU $\max(0, z)$")
ax.set_ylim(-1.5, 3)
ax.set_xlabel("$z$ (weighted sum + bias)")
ax.set_ylabel(r"activation $\sigma(z)$")
ax.legend()
plt.show()

Why is the nonlinearity essential? If every neuron were linear, stacking them would just give another linear function: a linear function of a linear function is linear. The nonlinear activation is what lets a network bend.

## 2. Layers and the multilayer perceptron

A **layer** is a set of neurons that all receive the same inputs. Each neuron has its own weights. A **multilayer perceptron (MLP)** feeds the outputs of one layer into the next:

$$\text{input } x \;\to\; \underbrace{\text{hidden layer}}_{H \text{ neurons}} \;\to\; \dots \;\to\; \text{output } \hat y.$$

The layers between input and output are called **hidden layers**. For one input, one hidden layer of $H$ neurons, and one output, the network is

$$\hat y(x) = b_2 + \sum_{k=1}^{H} v_k \tanh\big(w_k x + b_k\big).$$

Read it as: a sum of $H$ tanh "steps", each with its own position, steepness, and height. With enough of them you can build almost any smooth curve. (There's a theorem to this effect, the *universal approximation theorem*.) The price: $3H + 1$ parameters with no physical meaning.

To get a feel for it, here are a few networks with $H = 20$ and **random** parameters. No training yet; this just shows the kind of functions an MLP can represent.

A new NumPy trick: `x[:, None]` turns a 1D array of shape `(N,)` into a column of shape `(N, 1)`. Combining a column `(N, 1)` with a row `(H,)` in arithmetic **broadcasts** to a full table of shape `(N, H)`: entry `[i, k]` is computed from `x[i]` and `w[k]`. That computes all $H$ neurons for all $N$ inputs in one line, with no loop.

In [ ]:
def mlp_1d(x, w, b, v, b2):
    """One-hidden-layer tanh network: b2 + sum_k v_k tanh(w_k x + b_k)."""
    hidden = np.tanh(x[:, None] * w + b)     # shape (N, H)
    return hidden @ v + b2                   # weighted sum over the H neurons -> shape (N,)

x = np.linspace(-3, 3, 300)
H = 20
fig, ax = plt.subplots()
for i in range(4):
    w, b, v = rng.normal(0, 2, H), rng.normal(0, 2, H), rng.normal(0, 0.5, H)
    ax.plot(x, mlp_1d(x, w, b, v, 0.0), label=f"random network {i + 1}")
ax.set_xlabel("input $x$")
ax.set_ylabel(r"output $\hat y$")
ax.legend()
plt.show()

The `@` operator in `hidden @ v` is **matrix multiplication**. Here it is just a compact way to write "for each row $i$, compute $\sum_k$ `hidden[i, k] * v[k]`", the same as `(hidden * v).sum(axis=1)`.

## 3. Loss functions

Training means choosing the parameters to minimize a loss. The two you need:

- **Mean squared error** for regression (predicting a number), as in least squares:
$$L_\text{MSE} = \frac{1}{N}\sum_{i=1}^{N}(\hat y_i - y_i)^2 .$$
- **Binary cross-entropy** for two-class classification, where the network outputs a probability $p_i$ that example $i$ is in class 1 and the label is $y_i \in \{0, 1\}$:
$$L_\text{BCE} = -\frac{1}{N}\sum_{i=1}^{N}\Big[y_i \ln p_i + (1-y_i)\ln(1-p_i)\Big].$$
For a class-1 example only the first term survives, $-\ln p_i$, which is 0 when $p_i = 1$ (confident and right) and huge when $p_i \to 0$ (confident and wrong). As you saw last week, this is the negative log-likelihood of the labels.

## 4. Gradient descent

A network can have thousands to billions of parameters, so we need a minimizer that scales. **Gradient descent** is the simplest: repeatedly take a small step downhill.

$$w \;\leftarrow\; w - \eta\,\frac{dL}{dw}.$$

The step size $\eta$ (eta) is the **learning rate**, a hyperparameter. With many parameters, $dL/dw$ becomes the **gradient** $\nabla L$, the vector of partial derivatives $\partial L/\partial w_j$, which points uphill; we step against it.

Start in 1D with a loss we can see, $L(w) = (w-2)^2 + 1$, so $dL/dw = 2(w-2)$ and the minimum is at $w = 2$.

In [ ]:
def loss_1d(w):
    return (w - 2)**2 + 1

def grad_1d(w):
    return 2 * (w - 2)

def gradient_descent_1d(w0, eta, n_steps):
    path = [w0]
    for _ in range(n_steps):
        path.append(path[-1] - eta * grad_1d(path[-1]))
    return np.array(path)

w_grid = np.linspace(-2, 6, 200)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
for ax, eta in zip(axes, [0.1, 0.8, 1.05]):
    path = gradient_descent_1d(-1.0, eta, 10)
    ax.plot(w_grid, loss_1d(w_grid), "k-", lw=1)
    ax.plot(path, loss_1d(path), "o-", ms=4)
    ax.set_title(rf"$\eta$ = {eta}, final $w$ = {path[-1]:.2f}")
    ax.set_xlabel("parameter $w$")
axes[0].set_ylabel("loss $L(w)$")
axes[0].set_ylim(0, 20)
plt.show()

- **Small $\eta$** (0.1): steady but slow progress.
- **Larger $\eta$** (0.8): overshoots and zig-zags across the valley, but converges.
- **Too large $\eta$** (1.05): every step overshoots by more than the last, and the parameter runs away. A diverging loss (or `nan`) during training almost always means the learning rate is too big.

In more dimensions the landscape is usually a long, narrow valley, because the loss is much more sensitive to some parameters than others. Take $L(w_1, w_2) = w_1^2 + 10\,w_2^2$:

In [ ]:
def loss_2d(w):
    return w[0]**2 + 10 * w[1]**2

def grad_2d(w):
    return np.array([2 * w[0], 20 * w[1]])

W1, W2 = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-1.5, 1.5, 200))

fig, ax = plt.subplots(figsize=(7, 4))
ax.contour(W1, W2, W1**2 + 10 * W2**2, levels=np.logspace(-1, 1.5, 12), colors="gray", linewidths=0.7)
for eta in [0.01, 0.09]:
    w = np.array([-2.8, 1.0])
    path = [w]
    for _ in range(30):
        w = w - eta * grad_2d(w)
        path.append(w)
    path = np.array(path)
    ax.plot(path[:, 0], path[:, 1], "o-", ms=3, label=rf"$\eta$ = {eta}: $L$ = {loss_2d(w):.3f} after 30 steps")
ax.plot(0, 0, "k*", ms=12)
ax.set_xlabel("$w_1$")
ax.set_ylabel("$w_2$")
ax.legend(loc="lower right")
plt.show()

With a small $\eta$ the path crawls along the shallow $w_1$ direction. A larger $\eta$ moves faster but zig-zags across the steep $w_2$ direction, and slightly larger still ($\eta > 0.1$ here) it diverges. One learning rate can't suit both directions. Modern optimizers such as **Adam** (which you'll use in PyTorch) fix much of this by keeping a running average of the gradient (momentum) and adapting the step size separately for each parameter.

### Your turn 1

Using `grad_2d` above, run gradient descent from $(-2.8, 1.0)$ for $\eta$ = 0.01, 0.05, 0.09, 0.099, and 0.101. For each, count how many steps it takes until $L < 10^{-6}$ (stop after 2000 steps if it never gets there). Which $\eta$ is fastest? Explain the result for 0.101 using the $w_2$ part of the update, $w_2 \leftarrow w_2 - 20\eta\,w_2 = (1 - 20\eta)\,w_2$.

In [ ]:
# Your code here

## 5. A neural network in pure NumPy

Now let's train the one-hidden-layer network from Section 2 to learn $y = \sin x$ on $[-\pi, \pi]$, writing every step by hand. We need the gradient of the loss with respect to all $3H + 1$ parameters. With $\hat y_i = b_2 + \sum_k v_k a_{ik}$ and $a_{ik} = \tanh(z_{ik})$, $z_{ik} = w_k x_i + b_k$, the chain rule gives:

$$\frac{\partial L}{\partial \hat y_i} = \frac{2}{N}(\hat y_i - y_i) \equiv \delta_i,$$

$$\frac{\partial L}{\partial b_2} = \sum_i \delta_i, \qquad
\frac{\partial L}{\partial v_k} = \sum_i \delta_i\, a_{ik},$$

$$\frac{\partial L}{\partial z_{ik}} = \delta_i\, v_k\,(1 - a_{ik}^2) \equiv \Delta_{ik}
\quad\Big(\text{using } \tfrac{d}{dz}\tanh z = 1 - \tanh^2 z\Big),$$

$$\frac{\partial L}{\partial w_k} = \sum_i \Delta_{ik}\, x_i, \qquad
\frac{\partial L}{\partial b_k} = \sum_i \Delta_{ik}.$$

Notice the pattern: we compute derivatives starting at the output and work **backward** through the network, reusing each result for the layer before. This is called **backpropagation**. It is nothing more than the chain rule, organized efficiently.

In [ ]:
x_train = np.linspace(-np.pi, np.pi, 100)
y_train = np.sin(x_train)

H = 10
w = rng.normal(0, 1, H)              # input -> hidden weights
b = rng.normal(0, 1, H)              # hidden biases
v = rng.normal(0, 1 / np.sqrt(H), H) # hidden -> output weights
b2 = 0.0                             # output bias

def forward(x, w, b, v, b2):
    a = np.tanh(x[:, None] * w + b)  # hidden activations, shape (N, H)
    return a, a @ v + b2             # also return a: the backward pass needs it

In [ ]:
eta = 0.05
losses = []
for step in range(10000):
    a, y_hat = forward(x_train, w, b, v, b2)
    losses.append(np.mean((y_hat - y_train)**2))

    # backward pass (backpropagation)
    delta = 2 * (y_hat - y_train) / len(x_train)     # dL/dy_hat, shape (N,)
    grad_b2 = delta.sum()
    grad_v = (delta[:, None] * a).sum(axis=0)        # shape (H,)
    Delta = delta[:, None] * v * (1 - a**2)          # dL/dz, shape (N, H)
    grad_w = (Delta * x_train[:, None]).sum(axis=0)
    grad_b = Delta.sum(axis=0)

    # gradient-descent update
    w -= eta * grad_w
    b -= eta * grad_b
    v -= eta * grad_v
    b2 -= eta * grad_b2

print(f"loss at start: {losses[0]:.3f},  after {len(losses)} steps: {losses[-1]:.2e}")

The update `w -= eta * grad_w` is shorthand for `w = w - eta * grad_w` (it modifies the array in place).

Hand-derived gradients are easy to get wrong, so it's good practice to **check** one against a finite difference, the central-difference formula from notebook 07.

In [ ]:
def mse_loss(w, b, v, b2):
    return np.mean((forward(x_train, w, b, v, b2)[1] - y_train)**2)

# analytic gradient (recompute at the current parameters)
a, y_hat = forward(x_train, w, b, v, b2)
delta = 2 * (y_hat - y_train) / len(x_train)
grad_w = ((delta[:, None] * v * (1 - a**2)) * x_train[:, None]).sum(axis=0)

# numerical gradient with respect to w[0]
h = 1e-6
w_plus, w_minus = w.copy(), w.copy()
w_plus[0] += h
w_minus[0] -= h
numerical = (mse_loss(w_plus, b, v, b2) - mse_loss(w_minus, b, v, b2)) / (2 * h)
print(f"backprop: {grad_w[0]:.6e}   finite difference: {numerical:.6e}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].semilogy(losses)
axes[0].set_xlabel("gradient-descent step")
axes[0].set_ylabel("MSE loss")

x_plot = np.linspace(-2 * np.pi, 2 * np.pi, 400)
axes[1].plot(x_plot, np.sin(x_plot), "k--", label=r"$\sin x$")
axes[1].plot(x_plot, forward(x_plot, w, b, v, b2)[1], label="network")
axes[1].axvspan(-np.pi, np.pi, color="gray", alpha=0.15, label="training range")
axes[1].set_xlabel("$x$ (rad)")
axes[1].legend()
plt.show()

Inside the training range the network matches $\sin x$ well. Outside it, the network has no idea that sine is periodic: each tanh flattens out, so the network just levels off. **A neural network only knows what its training data shows it.** Keep this in mind; you'll see it again with the pendulum.

### Your turn 2

Retrain the NumPy network with $H = 2$ and with $H = 30$ hidden neurons (re-initialize the parameters each time, and keep everything else the same). Plot the three fits together. With $H = 2$, why can't the network represent one full period of $\sin x$? (Think about how many "steps" a sum of 2 tanh functions can make.)

In [ ]:
# Your code here

## 6. PyTorch: tensors and automatic differentiation

Writing backpropagation by hand for every network would be miserable. **PyTorch** does it for you. Its core object is the **tensor**, which is essentially a NumPy array with two extra powers: it can live on a GPU, and it can keep track of the operations applied to it so that derivatives can be computed automatically.

In [ ]:
import torch
from torch import nn

torch.manual_seed(14)                  # PyTorch has its own random generator; seed it too

t = torch.tensor([1.0, 2.0, 3.0])
print(t, t.dtype, t.shape)
print(t**2 + 1)                        # arithmetic works like NumPy
print(torch.sin(t).numpy())            # .numpy() converts back to a NumPy array

PyTorch uses 32-bit floats (`float32`, about 7 significant digits; see notebook 06) by default, while NumPy uses 64-bit. When converting NumPy data, ask for `float32` explicitly with `torch.tensor(array, dtype=torch.float32)`; mixing the two types causes errors.

### Autograd

If you create a tensor with `requires_grad=True`, PyTorch records every operation on it. Calling `.backward()` on a final scalar result then runs backpropagation and stores the derivative in the `.grad` attribute. This is called **automatic differentiation** (autograd). It is not finite differences (no step size $h$, no truncation error) and not symbolic algebra: it applies the chain rule to the actual sequence of operations, giving derivatives exact to rounding.

Test it on $f(x) = x^2\sin x$, whose derivative is $f'(x) = 2x\sin x + x^2\cos x$:

In [ ]:
x = torch.tensor(1.3, requires_grad=True)
f = x**2 * torch.sin(x)
f.backward()                     # compute df/dx

x0 = 1.3
print("autograd:", x.grad.item())
print("analytic:", 2 * x0 * np.sin(x0) + x0**2 * np.cos(x0))

`.item()` extracts a plain Python number from a one-element tensor.

Autograd also works on whole arrays. Since `.backward()` needs a single number, we sum the outputs first; because each output depends only on its own input, the derivative of the sum with respect to `x[i]` is $f'(x_i)$.

In [ ]:
x = torch.linspace(0, 2 * np.pi, 100, requires_grad=True)
f = torch.sin(x)
f.sum().backward()

fig, ax = plt.subplots()
ax.plot(x.detach().numpy(), f.detach().numpy(), label=r"$\sin x$")
ax.plot(x.detach().numpy(), x.grad.numpy(), label="autograd derivative")
ax.plot(x.detach().numpy(), np.cos(x.detach().numpy()), "k:", label=r"$\cos x$")
ax.set_xlabel("$x$ (rad)")
ax.legend()
plt.show()

`.detach()` gives a copy of a tensor that is no longer tracked by autograd; you need it before converting a tracked tensor to NumPy.

### Your turn 3

The Lennard-Jones potential models the interaction of two neutral atoms (for example, two argon atoms) at separation $r$:
$$U(r) = 4\varepsilon\left[\left(\frac{\sigma}{r}\right)^{12} - \left(\frac{\sigma}{r}\right)^{6}\right].$$
The repulsive $r^{-12}$ term keeps the atoms from overlapping; the attractive $r^{-6}$ term is the van der Waals attraction. Use units where $\varepsilon = \sigma = 1$. Use autograd to compute the force $F(r) = -dU/dr$ on a grid $r \in [0.95, 3]$, and compare it to the analytic derivative on a plot. Then find where $F = 0$ and check it against the analytic value $r = 2^{1/6}\sigma$.

In [ ]:
# Your code here

### CPU or GPU?

Neural-network training is mostly multiplying large tables of numbers, which graphics processors (GPUs) do extremely fast. PyTorch can run on a GPU if one is available. The standard idiom picks the device once:

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("using device:", device)

On Colab you can get a free GPU with **Runtime → Change runtime type → T4 GPU** (then re-run the notebook). Everything in this notebook is small and runs fine on a CPU, so you don't need one. To use a GPU, both the model and the data must be moved to it with `.to(device)`; the code below does this so that it works either way.

## 7. Building networks with `nn`

The `torch.nn` module provides the building blocks:

- `nn.Linear(n_in, n_out)`: a layer of `n_out` neurons, each computing a weighted sum of `n_in` inputs plus a bias (no activation). Its weights start random.
- `nn.Tanh()`, `nn.ReLU()`, `nn.Sigmoid()`: activation functions.
- `nn.Sequential(...)`: chains layers in order into one network.

Here is the same architecture as our NumPy network, but with 32 hidden neurons:

In [ ]:
model = nn.Sequential(
    nn.Linear(1, 32),   # input (1 number) -> 32 hidden neurons
    nn.Tanh(),
    nn.Linear(32, 1),   # 32 hidden -> 1 output
).to(device)
print(model)

n_params = sum(p.numel() for p in model.parameters())
print("number of parameters:", n_params)

`model.parameters()` yields every weight and bias tensor, and `.numel()` counts the numbers in each. The first layer has $32 \times 1$ weights + 32 biases, and the second has $1 \times 32$ weights + 1 bias, for 97 in total.

PyTorch layers expect a 2D input of shape `(N, n_features)`, just like scikit-learn. The training recipe has four steps per iteration, which you will see in every PyTorch program:

1. `optimizer.zero_grad()`: clear old gradients (PyTorch *adds* new gradients to `.grad`, so they must be reset).
2. `loss = loss_fn(model(X), Y)`: forward pass.
3. `loss.backward()`: backpropagation fills `.grad` for every parameter.
4. `optimizer.step()`: update the parameters using the gradients.

The optimizer here is `torch.optim.Adam`, and `lr` is its learning rate.

In [ ]:
X = torch.tensor(x_train[:, None], dtype=torch.float32).to(device)   # shape (100, 1)
Y = torch.tensor(y_train[:, None], dtype=torch.float32).to(device)

loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for step in range(1000):
    optimizer.zero_grad()
    loss = loss_fn(model(X), Y)
    loss.backward()
    optimizer.step()
print(f"final MSE: {loss.item():.2e}")

Ten lines replace all of our hand-written backpropagation, and Adam (with a somewhat bigger network) reaches a lower loss in a tenth as many steps as our plain gradient descent.

To make predictions after training, wrap the calculation in `with torch.no_grad():`, which tells PyTorch not to record operations for autograd (faster, and needed before calling `.numpy()`). `.cpu()` moves the result back from the GPU, if it was there.

In [ ]:
with torch.no_grad():
    y_pred = model(X).cpu().numpy().ravel()
print("max |error| on the training grid:", np.abs(y_pred - y_train).max().round(4))

### Defining a network as a class

Larger PyTorch programs usually define a network as a **class**. A class is a blueprint for making objects that bundle data with functions (*methods*). You won't need to write classes in this course, but you'll see this pattern constantly in PyTorch code, so here is how to read it:

In [ ]:
class SmallNet(nn.Module):
    def __init__(self, n_hidden):
        super().__init__()                     # required boilerplate
        self.hidden = nn.Linear(1, n_hidden)   # layers are stored as attributes
        self.output = nn.Linear(n_hidden, 1)

    def forward(self, x):                      # what happens when you call net(x)
        return self.output(torch.tanh(self.hidden(x)))

net = SmallNet(32)
print(net)

`__init__` runs when the object is created and sets up the layers; `forward` describes how an input flows through them. The advantage over `nn.Sequential` is flexibility: `forward` can contain any Python code (branches, loops, skip connections).

## 8. A physics regression: the period of a large-amplitude pendulum

### The physics

A simple pendulum of length $L$ released from rest at angle $\theta_0$ swings back and forth. For small angles, $\sin\theta \approx \theta$, the motion is simple harmonic, and the period doesn't depend on amplitude:

$$T_0 = 2\pi\sqrt{L/g}.$$

For large amplitudes the restoring force is weaker than the small-angle approximation assumes, and the period gets longer. The exact result (you met it in week 6) is

$$T(\theta_0) = 4\sqrt{\frac{L}{g}}\;K\!\left(\sin^2\frac{\theta_0}{2}\right),$$

where $K$ is the complete elliptic integral of the first kind. Watch out for a notorious convention trap: `scipy.special.ellipk(m)` takes the **parameter** $m = k^2$, not the modulus $k = \sin(\theta_0/2)$. As $\theta_0 \to \pi$ (the pendulum starts balanced upside down), $T \to \infty$.

### The data

We **simulate** a pendulum experiment: 400 measured periods for random release angles between 0.1 and 2.0 rad (6° to 115°), each with 5 ms of timing noise. The task: learn $T(\theta_0)$ from these data alone.

In [ ]:
from scipy.special import ellipk

L_pend = 1.0                          # pendulum length in m
g = constants.g                       # 9.80665 m/s^2
T0 = 2 * np.pi * np.sqrt(L_pend / g)  # small-angle period in s

def period_exact(theta0):
    """Exact pendulum period in s; theta0 in rad. Note ellipk takes m = k^2."""
    return 4 * np.sqrt(L_pend / g) * ellipk(np.sin(theta0 / 2)**2)

theta = rng.uniform(0.1, 2.0, size=400)                       # release angles in rad
period = period_exact(theta) + rng.normal(0, 0.005, size=400) # "measured" periods in s
print(f"T0 = {T0:.4f} s;  T(2.0 rad) = {period_exact(2.0):.4f} s")

Split the data into training (75%) and validation (25%) sets and convert to tensors of shape `(N, 1)`.

To train on **mini-batches** we use two PyTorch helpers:

- `TensorDataset(X, Y)` pairs up inputs and targets, so `dataset[i]` gives `(X[i], Y[i])`.
- `DataLoader(dataset, batch_size=32, shuffle=True)` hands out the data in random batches of 32. One pass through all the batches is called an **epoch**.

Why mini-batches? Computing the gradient on a random batch is much cheaper than on the whole data set, and the gradient is still about right on average. Taking many cheap, slightly noisy steps beats taking a few exact ones. This is **stochastic gradient descent**, and it's how essentially all neural networks are trained.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

th_train, th_val, T_train, T_val = train_test_split(theta, period, test_size=0.25, random_state=0)

def to_tensor(a):
    return torch.tensor(np.asarray(a).reshape(-1, 1), dtype=torch.float32).to(device)

X_train, Y_train = to_tensor(th_train), to_tensor(T_train)
X_val, Y_val = to_tensor(th_val), to_tensor(T_val)

train_loader = DataLoader(TensorDataset(X_train, Y_train), batch_size=32, shuffle=True)
print(len(train_loader.dataset), "training examples in", len(train_loader), "batches per epoch")

Here is a training function we'll reuse. After each epoch it records the average training loss and the loss on the validation set. `model.train()` and `model.eval()` switch some layer types between training and evaluation behavior; our simple layers don't care, but it's a good habit.

In [ ]:
def train(model, train_loader, X_val, Y_val, loss_fn, optimizer, n_epochs):
    history = {"train": [], "val": []}
    for epoch in range(n_epochs):
        model.train()
        total = 0.0
        for xb, yb in train_loader:              # loop over mini-batches
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
            total += loss.item() * len(xb)
        history["train"].append(total / len(train_loader.dataset))
        model.eval()
        with torch.no_grad():
            history["val"].append(loss_fn(model(X_val), Y_val).item())
    return history

In [ ]:
torch.manual_seed(0)
pend_model = nn.Sequential(
    nn.Linear(1, 32), nn.Tanh(),
    nn.Linear(32, 32), nn.Tanh(),
    nn.Linear(32, 1),
).to(device)
optimizer = torch.optim.Adam(pend_model.parameters(), lr=0.01)

hist = train(pend_model, train_loader, X_val, Y_val, nn.MSELoss(), optimizer, n_epochs=150)
print(f"final training MSE = {hist['train'][-1]:.2e} s^2,  validation MSE = {hist['val'][-1]:.2e} s^2")
print(f"noise variance = {0.005**2:.1e} s^2")

In [ ]:
fig, ax = plt.subplots()
ax.semilogy(hist["train"], label="training loss")
ax.semilogy(hist["val"], label="validation loss")
ax.axhline(0.005**2, color="gray", ls=":", label="noise variance")
ax.set_xlabel("epoch")
ax.set_ylabel(r"MSE (s$^2$)")
ax.legend()
plt.show()

These are **loss curves**, the first thing to look at whenever you train a network. Both losses fall and level off near the noise variance: the network has learned what there is to learn, and the validation loss is close to the training loss, so it isn't overfitting. (The jitter comes from the randomness of mini-batches.)

### Interpolation vs extrapolation

Now evaluate the trained network on angles from 0 to 3 rad, well beyond the training range of 0.1–2.0 rad.

In [ ]:
theta_plot = np.linspace(0.0, 3.0, 300)
with torch.no_grad():
    T_nn = pend_model(to_tensor(theta_plot)).cpu().numpy().ravel()

fig, ax = plt.subplots()
ax.plot(th_train, T_train, ".", ms=3, color="gray", label="training data")
ax.plot(theta_plot, period_exact(theta_plot), "k-", label="exact")
ax.plot(theta_plot, T_nn, "r--", label="neural network")
ax.axhline(T0, color="C0", ls=":", label="small-angle $T_0$")
ax.axvspan(0.1, 2.0, color="gray", alpha=0.12)
ax.set_ylim(1.8, 5)
ax.set_xlabel(r"release angle $\theta_0$ (rad)")
ax.set_ylabel("period $T$ (s)")
ax.legend(loc="upper left")
plt.show()

for th in [1.0, 2.5, 2.9]:
    with torch.no_grad():
        T_pred = pend_model(to_tensor([th])).item()
    print(f"theta0 = {th} rad:  network {T_pred:.3f} s,  exact {period_exact(th):.3f} s")

Inside the shaded training region the network is excellent. Outside it, the network continues smoothly but has no way of knowing that $T$ blows up as $\theta_0 \to \pi$. Physics knew this; the data didn't show it.

**Lesson:** a neural network is an interpolator. It is trustworthy only where it has seen data. A fit based on the right physics (here, the elliptic-integral formula) extrapolates; a generic flexible model does not. Notebook 18 shows one way to put physics back into a network.

### Your turn 4

(a) Retrain the pendulum network on simulated data covering 0.1–2.8 rad. Does it now capture the steep rise? How well does it do at 3.0 rad?
(b) Instead of $T$, train the network to predict the dimensionless ratio $T/T_0$. Explain why a network trained this way would work for a pendulum of any length $L$ (on any planet), while one trained on $T$ itself would not.

In [ ]:
# Your code here

## 9. Overfitting and early stopping

The pendulum network had plenty of data. What happens with a big network and only a few noisy points? Let's revisit the damped oscillation $x(t) = e^{-t/2}\cos(2\pi t)$ from last week: 20 training points and 20 validation points with noise 0.15, and a network with two hidden layers of 64 neurons (about 4,400 parameters, for 20 data points!). With so little data we use the whole training set as one batch.

In [ ]:
def true_signal(t):
    return np.exp(-t / 2) * np.cos(2 * np.pi * t)

t_all = rng.uniform(0, 2, size=40)                       # time in s
x_all = true_signal(t_all) + rng.normal(0, 0.15, size=40)
Xs_train, Ys_train = to_tensor(t_all[:20]), to_tensor(x_all[:20])
Xs_val, Ys_val = to_tensor(t_all[20:]), to_tensor(x_all[20:])

def make_big_net():
    return nn.Sequential(nn.Linear(1, 64), nn.Tanh(),
                         nn.Linear(64, 64), nn.Tanh(),
                         nn.Linear(64, 1)).to(device)

In [ ]:
torch.manual_seed(1)
big = make_big_net()
optimizer = torch.optim.Adam(big.parameters(), lr=0.003)
loss_fn = nn.MSELoss()

train_hist, val_hist = [], []
for epoch in range(3000):
    optimizer.zero_grad()
    loss = loss_fn(big(Xs_train), Ys_train)
    loss.backward()
    optimizer.step()
    with torch.no_grad():
        train_hist.append(loss.item())
        val_hist.append(loss_fn(big(Xs_val), Ys_val).item())

best_epoch = int(np.argmin(val_hist))
print(f"lowest validation loss {min(val_hist):.4f} at epoch {best_epoch}; "
      f"at the end: train {train_hist[-1]:.4f}, validation {val_hist[-1]:.4f}")

In [ ]:
fig, ax = plt.subplots()
ax.semilogy(train_hist, label="training loss")
ax.semilogy(val_hist, label="validation loss")
ax.axvline(best_epoch, color="k", ls=":", label=f"best epoch ({best_epoch})")
ax.axhline(0.15**2, color="gray", ls="--", label="noise variance")
ax.set_xlabel("epoch")
ax.set_ylabel("MSE")
ax.legend()
plt.show()

The classic overfitting signature: the training loss keeps falling far below the noise level (the network is memorizing the noise), while the validation loss reaches a minimum and then creeps back up. It's the same U-shape as the polynomial degree plot from last week, with training time playing the role of model complexity. (The occasional sharp spikes are the optimizer briefly overshooting; they are common and harmless here.)

**Early stopping** is the simplest remedy: monitor the validation loss, remember the parameters at its best value, and stop when it hasn't improved for a while (the **patience**). The parameters of a model are stored in its `state_dict()`; `copy.deepcopy` makes an independent copy so later training doesn't overwrite it.

In [ ]:
import copy

torch.manual_seed(1)
big = make_big_net()
optimizer = torch.optim.Adam(big.parameters(), lr=0.003)

best_val, best_state, patience, wait = np.inf, None, 200, 0
for epoch in range(3000):
    optimizer.zero_grad()
    loss_fn(big(Xs_train), Ys_train).backward()
    optimizer.step()
    with torch.no_grad():
        val = loss_fn(big(Xs_val), Ys_val).item()
    if val < best_val:
        best_val, best_state, wait = val, copy.deepcopy(big.state_dict()), 0
    else:
        wait += 1
        if wait >= patience:
            print(f"stopping at epoch {epoch}; best validation loss {best_val:.4f}")
            break

big.load_state_dict(best_state)       # restore the best parameters

In [ ]:
t_plot = np.linspace(0, 2, 300)
with torch.no_grad():
    x_early = big(to_tensor(t_plot)).cpu().numpy().ravel()

fig, ax = plt.subplots()
ax.plot(t_plot, true_signal(t_plot), "k--", label="true function")
ax.plot(t_all[:20], x_all[:20], "o", label="training data")
ax.plot(t_all[20:], x_all[20:], "s", mfc="none", label="validation data")
ax.plot(t_plot, x_early, label="network (early stopping)")
ax.set_xlabel("time $t$ (s)")
ax.set_ylabel("position $x$ (arb. units)")
ax.legend()
plt.show()

Other common remedies: get more data, use a smaller network, or add regularization. In PyTorch, the $\alpha\sum w^2$ penalty from last week is available as the `weight_decay` option of the optimizer, e.g. `torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)`.

Note that the validation set was used to *choose when to stop*, so it is part of training in a broad sense. To report honest final performance, you'd still need a separate test set.

## 10. Classification: an MLP on the MAGIC telescope data

Back to last week's gamma-ray telescope data. Quick recap: a cosmic gamma ray hitting the atmosphere starts a cascade of particles (an air shower), whose Cherenkov light a telescope images as an ellipse. Hadronic cosmic rays make similar but broader, more irregular images, and vastly outnumber the gammas. The ten Hillas parameters describe each image's shape, and the task is to separate gamma (signal, label 1) from hadron (background, label 0).

We load and split the data exactly as last week (same `random_state`, so the test set is the same).

In [ ]:
import os
DATA_URL = "https://raw.githubusercontent.com/burzynski-lab/OU_PHYS2222_ComputationalPhysics/main/Data/"

def data_path(filename):
    """Return a local path to a course data file if it exists, otherwise its GitHub URL."""
    for folder in ["../Data", "Data", "."]:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return path
    return DATA_URL + filename

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.ensemble import HistGradientBoostingClassifier

columns = ["fLength", "fWidth", "fSize", "fConc", "fConc1", "fAsym",
           "fM3Long", "fM3Trans", "fAlpha", "fDist", "class"]
magic = pd.read_csv(data_path("magic04.data"), header=None, names=columns)
X = magic[columns[:-1]].to_numpy()
y = (magic["class"] == "g").astype(int).to_numpy()

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
X_tr, X_va, y_tr, y_va = train_test_split(
    X_train_full, y_train_full, test_size=0.15, random_state=0, stratify=y_train_full)
print("train / validation / test:", len(y_tr), len(y_va), len(y_test))

Neural networks are trained by gradient descent, and they train badly when inputs span very different scales (`fSize` is about 3, `fDist` is about 200). So standardize, fitting the scaler on the training set only.

In [ ]:
scaler = StandardScaler().fit(X_tr)

def to_tensors(X, y):
    Xt = torch.tensor(scaler.transform(X), dtype=torch.float32).to(device)
    yt = torch.tensor(y.reshape(-1, 1), dtype=torch.float32).to(device)
    return Xt, yt

Xt_tr, yt_tr = to_tensors(X_tr, y_tr)
Xt_va, yt_va = to_tensors(X_va, y_va)
Xt_te, yt_te = to_tensors(X_test, y_test)
magic_loader = DataLoader(TensorDataset(Xt_tr, yt_tr), batch_size=128, shuffle=True)

The network has 10 inputs, two hidden layers of 64 ReLU neurons, and 1 output. The output is a raw number $z$ (called a **logit**) rather than a probability; the probability is $p = \text{sigmoid}(z)$. We use `nn.BCEWithLogitsLoss`, which applies the sigmoid and the binary cross-entropy together in one numerically safer step (computing $\ln p$ when $p$ has rounded to 0 gives $-\infty$; the combined formula avoids that).

In [ ]:
torch.manual_seed(42)
mlp = nn.Sequential(
    nn.Linear(10, 64), nn.ReLU(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, 1),
).to(device)
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3)

magic_hist = train(mlp, magic_loader, Xt_va, yt_va, nn.BCEWithLogitsLoss(), optimizer, n_epochs=30)

fig, ax = plt.subplots()
ax.plot(magic_hist["train"], label="training loss")
ax.plot(magic_hist["val"], label="validation loss")
ax.set_xlabel("epoch")
ax.set_ylabel("binary cross-entropy")
ax.legend()
plt.show()

In [ ]:
with torch.no_grad():
    p_mlp = torch.sigmoid(mlp(Xt_te)).cpu().numpy().ravel()

acc = np.mean((p_mlp > 0.5) == y_test)
print(f"MLP test accuracy = {acc:.3f},  AUC = {roc_auc_score(y_test, p_mlp):.4f}")

Now the comparison you've been waiting for: gradient-boosted trees, trained on the same training set (trees don't need scaling).

In [ ]:
gbt = HistGradientBoostingClassifier(random_state=0).fit(X_train_full, y_train_full)
p_gbt = gbt.predict_proba(X_test)[:, 1]

fig, ax = plt.subplots(figsize=(6, 5.5))
for name, p in [("MLP (PyTorch)", p_mlp), ("gradient-boosted trees", p_gbt)]:
    fpr, tpr, _ = roc_curve(y_test, p)
    ax.plot(tpr, 1 - fpr, label=f"{name}, AUC = {roc_auc_score(y_test, p):.4f}")
ax.set_xlabel("signal efficiency")
ax.set_ylabel("background rejection")
ax.legend(loc="lower left")
plt.show()

The neural network is good, but the boosted trees, trained in a couple of seconds with default settings and no feature scaling, are at least as good. This is typical: on **tabular data** (a modest number of hand-crafted features per event, like Hillas parameters), gradient-boosted trees are very hard to beat and much easier to train. Neural networks shine when the inputs are raw and high-dimensional with structure the network can exploit: images, sequences, point clouds of detector hits, the whole shower image instead of ten numbers summarizing it. You'll see an example next week.

The practical advice: **start with a simple baseline** (logistic regression, then boosted trees) before reaching for a neural network, and only keep the network if it clearly beats them on the validation set.

### Your turn 5

Try to make the MLP beat the boosted trees. Ideas: more or wider hidden layers, more epochs (watch the validation loss for overfitting!), a different learning rate, `weight_decay` in `torch.optim.AdamW`, or a larger batch size. Keep a record of what you tried and the validation AUC for each (don't tune on the test set). How close do you get, and how much more effort was it than the trees?

In [ ]:
# Your code here

## Summary

- A **neuron** computes $\sigma(\mathbf{w}\cdot\mathbf{x} + b)$; common activations are sigmoid, tanh, and ReLU. Without nonlinear activations, stacking layers gives nothing new.
- A **multilayer perceptron** stacks layers of neurons; it's a flexible function with many parameters and no built-in physics.
- Losses: **MSE** for regression, **binary cross-entropy** (negative log-likelihood) for classification.
- **Gradient descent**: $w \leftarrow w - \eta\,\nabla L$. Too small an $\eta$ is slow; too large diverges. **Adam** adapts the step for each parameter.
- **Backpropagation** is the chain rule applied from the output backward. Check hand-coded gradients with finite differences.
- PyTorch: `torch.tensor(..., dtype=torch.float32)`, `requires_grad=True`, `.backward()`, `.grad`, `.detach()`, `.item()`, `torch.no_grad()`, `torch.manual_seed`, `device = "cuda" if torch.cuda.is_available() else "cpu"`, `.to(device)`.
- Building and training: `nn.Linear`, `nn.ReLU`/`nn.Tanh`, `nn.Sequential`, `nn.MSELoss`, `nn.BCEWithLogitsLoss`, `torch.optim.Adam`, `TensorDataset`, `DataLoader(batch_size, shuffle=True)`; the loop `zero_grad` → forward → `backward` → `step`.
- Always plot **training and validation loss curves**. A rising validation loss means overfitting; **early stopping** keeps the best `state_dict`.
- Neural networks **interpolate** well but **do not extrapolate** beyond their training data.
- On tabular data, **gradient-boosted trees** are often as good as or better than neural networks. Start with simple baselines.

## Further reading

- PyTorch tutorials ("Learn the Basics"): https://pytorch.org/tutorials/beginner/basics/intro.html
- M. Nielsen, *Neural Networks and Deep Learning*, free online book with a clear explanation of backpropagation: http://neuralnetworksanddeeplearning.com/
- P. Mehta et al., "A high-bias, low-variance introduction to Machine Learning for physicists", *Physics Reports* 810 (2019), https://arxiv.org/abs/1803.08823
- 3Blue1Brown, *Neural networks* video series (visual intuition for gradient descent and backpropagation).